# RoBacTutor — Ask Mode (free questions)
### MSc Computer Science Dissertation — University College Birmingham
**Author:** Vasile Bria | **Student ID:** BRI23222497
**Supervisor:** Farah Shahid
**Component:** free-form student Q&A, retrieval-grounded

---
This notebook is standalone -- it loads its own model/adapter/RAG index, separate
from the Testeaza-te notebook. It exists to answer one question empirically:
**does Ask mode need generation at all, and if so, adapter or base model?**

It builds THREE answer strategies so you can compare them side by side on the
same questions:

1. **Extractive (no generation)** -- just returns the top retrieved textbook
   passage(s) directly. Zero hallucination risk, since nothing is generated.
   This mirrors the pivot already validated for grading: show real content
   instead of trusting the model to synthesize it correctly.
2. **Generative, adapter enabled** -- RAG context + the fine-tuned adapter.
3. **Generative, adapter disabled (base model)** -- RAG context + base RoMistral.

Given everything found so far (generic-template hallucination on istorie,
self-answering + fabricated citation on matematica, both independent of
retrieval quality per the dissertation's own finding), it's very plausible
option 1 ends up being the one that actually ships. Options 2/3 are here so
you have the comparison as evidence either way, not just an assumption.

## Instructions
1. Set runtime to **T4 GPU**
2. Run Cells 1-6 in order
3. Upload `robactutor_lora_adapters_BRI23222497.zip` when prompted (Cell 3)
4. Upload `robactutor_rag_BRI23222497.zip` when prompted (Cell 5) -- use the
   correct 8,496-chunk extended version, not an older one
5. Cell 8 runs all three strategies across all 4 subjects for direct comparison
6. Cell 9 is a simple interactive tester for follow-up questions of your choosing

## Cell 1 — Install
> Unpinned/latest versions -- avoids the transformers/bitsandbytes frozenset bug hit earlier with pinned 4.46.0.
> Runtime restarts after this. Skip after restart.

In [ ]:
# RoBacTutor Ask Mode — Cell 1: Install
# Vasile Bria | BRI23222497 | UCB 2025-2026

!pip install -q -U \
    transformers \
    bitsandbytes \
    peft \
    accelerate \
    sentencepiece \
    sentence-transformers \
    faiss-cpu

print("All packages installed!")
print("Restarting...")
import os
os.kill(os.getpid(), 9)


## Cell 2 — Verify Environment

In [ ]:
# RoBacTutor Ask Mode — Cell 2: Verify
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch
import transformers
import peft
import sentence_transformers
import faiss

print("=" * 50)
print("RoBacTutor Ask Mode")
print("Vasile Bria | BRI23222497 | UCB")
print("=" * 50)
print(f"transformers:         {transformers.__version__}")
print(f"peft:                 {peft.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"FAISS:                {faiss.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️  No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run from Cell 1.")
print("\n✅ Environment ready")


## Cell 3 — Upload Adapter and Load Model
> Upload `robactutor_lora_adapters_BRI23222497.zip` when prompted.
> Takes 1-2 minutes to load the base model + adapter in 4-bit.

In [ ]:
# RoBacTutor Ask Mode — Cell 3: Load fine-tuned model
# Vasile Bria | BRI23222497 | UCB 2025-2026

from google.colab import files
import zipfile, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Upload robactutor_lora_adapters_BRI23222497.zip...")
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]

ADAPTER_DIR = "./robactutor-adapters"
os.makedirs(ADAPTER_DIR, exist_ok=True)
with zipfile.ZipFile(zip_file, "r") as zf:
    zf.extractall(ADAPTER_DIR)
print(f"Extracted to {ADAPTER_DIR}")
print("Files:", os.listdir(ADAPTER_DIR))

MODEL_ID = "OpenLLM-Ro/RoMistral-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("\nLoading base model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n✅ Model + adapter loaded! VRAM used: {used:.1f} / {total:.1f} GB")


## Cell 4 — Generation Helper
> One function, with a switch for adapter on/off, so the exact same code path
> is used for both comparisons (no risk of divergent prompt-building logic
> between the two conditions).

In [ ]:
# RoBacTutor Ask Mode — Cell 4: Generation helper
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch

def generate_answer(system_prompt: str, instruction: str, use_adapter: bool = True,
                     max_new_tokens: int = 400, temperature: float = 0.3) -> str:
    prompt = f"<s>[INST] {system_prompt}\n\n{instruction} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    def _run():
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if use_adapter:
        return _run()
    else:
        # model.disable_adapter() -- NOT base_model.generate() directly.
        # PEFT modifies the base model's layers in-place, so calling base_model
        # would still run adapter-modified weights. This was a real bug caught
        # earlier in the Testeaza-te notebook (Cell 10).
        with model.disable_adapter():
            return _run()

print("✅ generate_answer() ready")


## Cell 5 — Upload RAG Index and Load Retrieval
> Upload `robactutor_rag_BRI23222497.zip` (the correct 8,496-chunk extended
> version) when prompted.

In [ ]:
# RoBacTutor Ask Mode — Cell 5: Load RAG index
# Vasile Bria | BRI23222497 | UCB 2025-2026

from google.colab import files
import zipfile, json, faiss
from sentence_transformers import SentenceTransformer

print("Upload robactutor_rag_BRI23222497.zip...")
uploaded = files.upload()
rag_zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(rag_zip_name, "r") as zf:
    zf.extractall(".")

rag_index = faiss.read_index("robactutor_faiss.index")
with open("robactutor_chunks.json", encoding="utf-8") as f:
    chunk_metadata = json.load(f)

rag_embedder = SentenceTransformer("all-MiniLM-L6-v2")  # MUST match index-build embedder

print(f"Loaded FAISS index: {rag_index.ntotal} vectors")
print(f"Loaded {len(chunk_metadata)} chunks")
subjects = {}
for c in chunk_metadata:
    subjects[c["subject"]] = subjects.get(c["subject"], 0) + 1
print("Subject breakdown:", subjects)


## Cell 6 — Retrieval Functions

In [ ]:
# RoBacTutor Ask Mode — Cell 6: Retrieval
# Vasile Bria | BRI23222497 | UCB 2025-2026

def retrieve(query: str, subject: str = None, source_type: str = None, top_k: int = 3) -> list[dict]:
    query_embedding = rag_embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, indices = rag_index.search(query_embedding, top_k * 5)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        meta = chunk_metadata[idx]
        if subject and meta["subject"] != subject:
            continue
        if source_type and meta.get("source_type") != source_type:
            continue
        results.append({
            "text": meta["text"],
            "subject": meta["subject"],
            "filename": meta.get("filename", "necunoscut"),
            "score": float(score),
        })
        if len(results) >= top_k:
            break
    return results


def get_reference_material(query: str, subject: str, top_k: int = 3) -> list[dict]:
    retrieved = retrieve(query, subject=subject, source_type="manual", top_k=top_k)
    if not retrieved:
        retrieved = retrieve(query, subject=subject, top_k=top_k)
    return retrieved

print("✅ retrieve() and get_reference_material() ready")


## Cell 7 — Three Answer Strategies
> `answer_extractive`: no generation, just the retrieved passages -- the safe default.
> `answer_generative`: RAG context + generation, adapter on/off via parameter.

In [ ]:
# RoBacTutor Ask Mode — Cell 7: Answer strategies
# Vasile Bria | BRI23222497 | UCB 2025-2026

ASK_SYSTEM_PROMPT = (
    "Esti un tutor pentru Bacalaureatul din Republica Moldova. "
    "Raspunde folosind DOAR informatia din materialul de referinta de mai jos. "
    "Daca materialul nu contine raspunsul, spune ca nu ai suficiente informatii."
)

def answer_extractive(subject: str, question: str, top_k: int = 2) -> dict:
    """No generation at all -- just the most relevant real textbook passage(s).
    Zero hallucination risk since nothing is synthesized."""
    reference = get_reference_material(question, subject, top_k=top_k)
    return {
        "strategy": "extractive",
        "answer": None,  # no generated answer -- reference IS the answer
        "reference": reference,
    }


def answer_generative(subject: str, question: str, use_adapter: bool = True, top_k: int = 3) -> dict:
    reference = get_reference_material(question, subject, top_k=top_k)
    context = "\n\n".join(r["text"] for r in reference)
    instruction = f"Material de referinta:\n{context}\n\nIntrebare: {question}"
    answer = generate_answer(ASK_SYSTEM_PROMPT, instruction, use_adapter=use_adapter)
    return {
        "strategy": "generative_adapter" if use_adapter else "generative_base",
        "answer": answer,
        "reference": reference,
    }

print("✅ answer_extractive() and answer_generative() ready")


## Cell 8 — Compare All Three Strategies Across All 4 Subjects
> This is the actual test. Read the generative outputs against the printed
> reference passages: is the answer genuinely grounded in that text, or does
> it drift into confident-sounding invented content (as grading did)?

In [ ]:
# RoBacTutor Ask Mode — Cell 8: Full comparison
# Vasile Bria | BRI23222497 | UCB 2025-2026

test_questions = {
    "matematica": "Cum calculez probabilitatea unui eveniment?",
    "istorie": "Ce a fost Pactul Ribbentrop-Molotov?",
    "limba_romana": "Ce este o figura de stil numita metafora?",
    "limba_engleza": "What is the difference between present perfect and past simple?",
}

for subject, question in test_questions.items():
    print("=" * 70)
    print(f"SUBJECT: {subject}")
    print(f"Q: {question}")

    extractive = answer_extractive(subject, question)
    print("\n--- EXTRACTIVE (no generation) ---")
    if not extractive["reference"]:
        print("  (niciun pasaj relevant gasit)")
    for r in extractive["reference"]:
        print(f"  - {r['filename']} (score: {r['score']:.3f})")
        print(f"    {r['text'][:300]}")

    gen_adapter = answer_generative(subject, question, use_adapter=True)
    print("\n--- GENERATIVE, ADAPTER ENABLED ---")
    print(gen_adapter["answer"])

    gen_base = answer_generative(subject, question, use_adapter=False)
    print("\n--- GENERATIVE, ADAPTER DISABLED (base model) ---")
    print(gen_base["answer"])

    print()


## Cell 9 — Interactive Tester
> Try your own follow-up questions once you've seen the Cell 8 comparison.

In [ ]:
# RoBacTutor Ask Mode — Cell 9: Interactive test
# Vasile Bria | BRI23222497 | UCB 2025-2026

subject = input("Alege materia (limba_romana / limba_engleza / istorie / matematica): ").strip()
question = input("Scrie intrebarea ta: ").strip()
strategy = input("Strategie (extractive / adapter / base): ").strip().lower()

if strategy == "extractive":
    result = answer_extractive(subject, question)
    print("\n--- RASPUNS (extractive) ---")
    for r in result["reference"]:
        print(f"\n- {r['filename']} (score: {r['score']:.3f})")
        print(f"  {r['text'][:400]}")
elif strategy == "base":
    result = answer_generative(subject, question, use_adapter=False)
    print("\n--- RASPUNS (generative, base model) ---")
    print(result["answer"])
else:
    result = answer_generative(subject, question, use_adapter=True)
    print("\n--- RASPUNS (generative, adapter) ---")
    print(result["answer"])
